# Week 38: Language Modelling

In [1]:
import math
from collections import Counter
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

## Shared data and vocabulary

In [2]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()
df_train_ko = df_train[df_train['lang'] == 'ko']
df_val_ko = df_validation[df_validation['lang'] == 'ko']

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased", use_fast=True
)

def tokenizeQuestion(df):
    return [tokenizer.tokenize(question) for question in df['question']]

t_questions = tokenizeQuestion(df_train_ko)
vocabulary = Counter(token for question in t_questions for token in question)
vocab = {token for token, count in vocabulary.items() if count >= 4}
vocab.update(['<s>', '</s>', '<unk>'])

def applyUnk(tokenizedQuestions, vocab):
    return [[token if token in vocab else '<unk>' for token in q]
            for q in tokenizedQuestions]

t_questions_filtered = applyUnk(t_questions, vocab)
val_unk = applyUnk(tokenizeQuestion(df_val_ko), vocab)

print('Training questions:', len(t_questions_filtered))
print('Validation questions:', len(val_unk))
print('Vocabulary size (including special tokens):', len(vocab))

Training questions: 2422
Validation questions: 356
Vocabulary size (including special tokens): 920


## Unigram baseline

In [3]:
unigram_counts = Counter(
    token for question in t_questions_filtered for token in question + ['</s>']
)
unigram_total = sum(unigram_counts.values())

def unigram_probability(token):
    return unigram_counts[token] / unigram_total

def unigram_perplexity(questions):
    log_probability = 0
    token_count = 0
    for question in questions:
        for token in question + ['</s>']:
            log_probability += math.log(unigram_probability(token))
            token_count += 1
    return math.exp(-log_probability / token_count)

unigram_ppl = unigram_perplexity(val_unk)
print(f'Unigram validation perplexity: {unigram_ppl:.2f}')

Unigram validation perplexity: 185.07


## Smoothed bigram

In [4]:
bigram_counts = Counter()
context_counts = Counter()

for question in t_questions_filtered:
    tokens = ['<s>'] + question + ['</s>']
    for previous, token in zip(tokens, tokens[1:]):
        bigram_counts[(previous, token)] += 1
        context_counts[previous] += 1

def bigram_probability(token, previous):
    return (bigram_counts[(previous, token)] + 1) / (context_counts[previous] + len(vocab))

def bigram_perplexity(questions):
    log_probability = 0
    token_count = 0
    for question in questions:
        tokens = ['<s>'] + question + ['</s>']
        for previous, token in zip(tokens, tokens[1:]):
            log_probability += math.log(bigram_probability(token, previous))
            token_count += 1
    return math.exp(-log_probability / token_count)

bigram_ppl = bigram_perplexity(val_unk)
print(f'Smoothed bigram validation perplexity: {bigram_ppl:.2f}')

Smoothed bigram validation perplexity: 52.21


## High- and low-perplexity questions

In [5]:
question_results = df_val_ko[['question']].copy()
question_results['unigram'] = [unigram_perplexity([q]) for q in val_unk]
question_results['bigram'] = [bigram_perplexity([q]) for q in val_unk]

print('Unigram: lowest perplexity')
print(question_results.sort_values('unigram').head(2).to_string(index=False))
print('Unigram: highest perplexity')
print(question_results.sort_values('unigram', ascending=False).head(2).to_string(index=False))

print('Bigram: lowest perplexity')
print(question_results.sort_values('bigram').head(2).to_string(index=False))
print('Bigram: highest perplexity')
print(question_results.sort_values('bigram', ascending=False).head(2).to_string(index=False))

Unigram: lowest perplexity
             question   unigram    bigram
  세상에서 가장 넓은 산은 무엇인가? 57.857549 10.657639
2011년  이란의 지도자는 누구인가? 65.407591  9.496725
Unigram: highest perplexity
                 question    unigram     bigram
부커 T. 워싱턴은 총 몇번의 결혼을 했나요? 716.071976 139.816802
부커 T. 워싱턴은 총 몇번의 결혼을 했나요? 716.071976 139.816802
Bigram: lowest perplexity
             question   unigram   bigram
 세상에서 가장 큰 대학교는 무엇인가? 73.347180 8.070707
2011년  이란의 지도자는 누구인가? 65.407591 9.496725
Bigram: highest perplexity
                                              question    unigram     bigram
임시정부는 종전의 정부가 무너진 후, 무정부 상태를 해소하기 위해 임시로 구성된 정부를 말하나요? 353.568750 298.249706
                       인체가 자외선에 많이 노출된다면 어떤 변화가 일어나는가? 286.002112 291.955630


## Feedforward neural trigram

In [6]:
import torch
import torch.nn as nn

CONTEXT_SIZE = 2

def buildWordToIx(vocab):
    return {tok: i for i, tok in enumerate(sorted(vocab))}

def sentenceToIds(tokens, wordToIx, contextSize=CONTEXT_SIZE):
    padded = ['<s>'] * contextSize + tokens + ['</s>']
    return [wordToIx[tok] for tok in padded]

def buildContextTargetPairs(tokenizedQuestionsUnk, wordToIx, contextSize=CONTEXT_SIZE):
    pairs = []
    for tokens in tokenizedQuestionsUnk:
        ids = sentenceToIds(tokens, wordToIx, contextSize)
        for i in range(contextSize, len(ids)):
            pairs.append((ids[i - contextSize:i], ids[i]))
    return pairs

In [7]:
class FFLM(nn.Module):
    def __init__(self, vocab_size, context_size=CONTEXT_SIZE, embed_dim=16, hidden_dim=32):
        super().__init__()
        self.context_size = context_size
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.hidden = nn.Linear(context_size * embed_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, context_ids):
        embedded = self.embedding(context_ids)
        flattened = embedded.reshape(embedded.size(0), -1)
        hidden = torch.tanh(self.hidden(flattened))
        return self.output(hidden)

In [8]:
def trainFeedforwardLm(model, pairs, epochs=300, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    lossFn = nn.CrossEntropyLoss()
    contexts = torch.tensor([p[0] for p in pairs])
    targets = torch.tensor([p[1] for p in pairs])

    for epoch in range(epochs):
        optimizer.zero_grad()
        logits = model(contexts)
        loss = lossFn(logits, targets)
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 100 == 0:
            print(f'  epoch {epoch+1:>4}  loss = {loss.item():.3f}')

In [9]:
@torch.no_grad()
def corpusPerplexityFeedforward(model, tokenizedQuestionsUnk, wordToIx, contextSize=CONTEXT_SIZE):
    model.eval()
    pairs = buildContextTargetPairs(tokenizedQuestionsUnk, wordToIx, contextSize)
    contexts = torch.tensor([p[0] for p in pairs])
    targets = torch.tensor([p[1] for p in pairs])

    logits = model(contexts)
    logProbs = torch.log_softmax(logits, dim=-1)
    tokenLogProbs = logProbs[range(len(targets)), targets]

    model.train()
    return math.exp(-tokenLogProbs.sum().item() / len(targets))

In [10]:
wordToIx = buildWordToIx(vocab)
torch.manual_seed(0)
model = FFLM(vocab_size=len(vocab))
pairs = buildContextTargetPairs(t_questions_filtered, wordToIx)

print(f'Training feedforward LM (context size={CONTEXT_SIZE})...')
trainFeedforwardLm(model, pairs, epochs=300, lr=0.01)

ppl = corpusPerplexityFeedforward(model, val_unk, wordToIx)
print(f'Feedforward LM validation perplexity: {ppl:.2f}')

Training feedforward LM (context size=2)...
  epoch  100  loss = 2.514
  epoch  200  loss = 2.003
  epoch  300  loss = 1.777
Feedforward LM validation perplexity: 25.44
